# Transformata Fouriera dla obrazów cyfrowych. Filtracja w dziedzinie częstotliwości.

## Cel ćwiczenia

- Zapoznanie z wykorzystaniem transformaty Fouriera w przetwarzaniu obrazów cyfrowych.
- Zapoznanie z pojęciem F-obrazu (amplitudy i fazy).
- Zapoznanie z własnościami transformaty Fouriera.
- Zapoznanie z filtracją w dziedzinie częstotliwości.

Na jednym z poprzednich ćwiczeń zetknęliśmy się z pojęciem konwolucji.
Przykładem może być filtracja dolno i górnoprzepustowa.
Operacja ta odpowiada mnożeniu w dziedzinie częstotliwości zgodnie z zależnością:

\begin{equation}
\mathcal{F}(g(x,y)*h(x,y)) = \mathcal{F}(g(x,y)) \cdot \mathcal{F}(h(x,y))
\tag{1}
\end{equation}

gdzie: $\mathcal{F}$ oznacza transformatę Fouriera, a $*$ jest splotem.

Operacja filtracji w dziedzinie częstotliwości może okazać się bardziej efektywna, jeżeli operacje $fft$ i $ifft$ (odpowiednio szybka transformata Fouriera -- *fast Fourier transform* -- oraz odwrotna szybka transformata Fouriera -- *inverse fast Fourier transform*) zajmą mniej czasu niż klasyczna konwolucja (zazwyczaj ma to miejsce dla dużego obrazu, z dużą maską).

Sama filtracja w dziedzinie częstotliwości to mnożenie punktowe całego obrazu przez jedną maskę.

W przypadku filtracji w dziedzinie częstotliwości zakłada się, że obraz "zawija się" na brzegach, co może powodować pewne artefakty (zostanie to pokazane w trakcie ćwiczenia).

W dziedzinie częstotliwości "działają" tylko filtry liniowe.
Filtry medianowe, maksymalne, minimalne itp. nie mają swoich odpowiedników.

## Dwuwymiarowa transformata Fouriera

1. Wczytaj plik "dwieFale.bmp" w skali szarości.
Jest to obraz powstały na podstawie następującej zależności:

\begin{equation}
L(m, n) = 128 + 127 \cdot \cos(\frac{2\pi m}{32}+\frac{3\pi}{4}) \cdot \cos(\frac{2\pi n}{8}-\frac{\pi}{2})
\tag{2}
\end{equation}

gdzie: $m$ i $n$ są odpowiednio numerami wierszy i kolumn.

2. Do realizacji dwuwymiarowej transformaty Fouriera służy funkcja `cv2.dft`.
Ustaw flagę `flags=cv2.DFT_COMPLEX_OUTPUT`.
Wykonaj transformatę na wczytanym obrazie.
W ten sposób uzyskuje się tzw. F-obraz.

3. Najniższe częstotliwości znajdują się w lewym-górnym rogu obrazu.
Dla celów wizualizacji (ale też przetwarzania) często wykonuje się tzw. przesunięcie F-obrazu, które powoduje, że niskie częstotliwości przesuwane są do środka obrazu.
Wykorzystaj funkcję `np.fft.fftshift`.
Jako pierwszy argument podaj wynik transformaty Fouriera.
Jako drugi argument podaj numery osi, wzdłuż których należy wykonać operację.
Pierwsza oś odnosi się do wierszy obrazu.
Druga oś odnosi się do kolumn obrazu.
Trzecia oś to część rzeczywista (`[:, :, 0]`) lub urojona (`[:, :, 1]`).
W naszym przypadku argument powinien wyglądać tak `[0,1]`.

4. Wyświetl wynik transformaty.
Na wspólnym wykresie umieść obraz oryginalny, amplitudę i fazę F-obrazu.
Amplitudę i fazę wyznacz za pomocą funkcji `cv2.cartToPolar`.
Pierwszym argumentem funkcji jest część rzeczywista wyniku, a drugim urojona.
Uwaga. W razie wątpliwości proszę sprawdzić rozmiary rezultatu transformaty Fouriera oraz przesunięcia.

5. Dla wizualizacji oblicz logarytm dziesiętny amplitudy: `ALog = np.log10(A + 1)`.
Wyświetl go zamiast amplitudy na poprzednim wykresie.

6. Wczytaj obrazy *kolo.bmp*, *kwadrat.bmp*, *kwadrat45.bmp*, *trojkat.bmp*.
Czy analizując F-obraz można coś powiedzieć o kierunku krawędzi obiektów?

7. Sprawdź (empirycznie) poprawność stwierdzenia:

`Dwuwymiarowa transformata Fouriera jest złożeniem dwóch transformat jednowymiarowych (wykonanych np. najpierw wierszowo, a później kolumnowo).` 
Jednowymiarowa transformata realizowana jest za pomocą funkcji fft (z bibloteki Numpy).

Wykonaj najpierw transformatę po wierszach: `FRow = np.fft.fft(I, axis=0)`.
Następnie po kolumnach: `FCol = np.fft.fft(FRow, axis=1)`.
Numpy zwraca wynik jako tablicę liczb zespolonych.
Część rzeczywistą można otrzymać w następujący sposób: `FCol.real`, a urojoną: `FCol.imag`.
Porównaj tak uzyskany wynik z rezultatem działania funkcji `cv2.dft`.
Można to zrobić wizualnie lub z wykorzystaniem funkcji `cv2.absdiff`.

In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import math
import os
import ssl
from urllib.request import urlretrieve

ssl._create_default_https_context = ssl._create_unverified_context

base_url = "https://raw.githubusercontent.com/vision-agh/poc_sw/master/08_Fourier/"
files = [
    "dwieFale.bmp",
    "kolo.bmp",
    "kwadrat.bmp",
    "kwadrat45.bmp",
    "kwadratKL.bmp",
    "kwadratS.bmp",
    "kwadratT.bmp",
    "lena.bmp",
    "trojkat.bmp",
    "literki.bmp",
    "wzorA.bmp",
]

for fname in files:
    if not os.path.exists(fname):
        urlretrieve(base_url + fname, fname)

I_Fale = cv2.imread("dwieFale.bmp", cv2.IMREAD_GRAYSCALE)

figFale, axsFale = plt.subplots()
axsFale.imshow(I_Fale, "gray", vmin=0, vmax=256)
axsFale.axis("off")
plt.show()

for image in files:
    img = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
    fig, axs = plt.subplots()
    axs.imshow(img, "gray", vmin=0, vmax=256)
    axs.axis("off")
    plt.show()

In [ ]:
def plot_3(image1, image2, image3,
           title1="Obraz 1", title2="Obraz 2", title3="Obraz 3"):    
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(image1, "gray", vmin=0, vmax=256)
    axs[0].set_title(title1)
    axs[0].axis("off")
    axs[1].imshow(image2, "gray")
    axs[1].set_title(title2)
    axs[1].axis("off")
    axs[2].imshow(image3, "gray")
    axs[2].set_title(title3)
    axs[2].axis("off")
    plt.show()

def fourier_not_log(image):
    image_float = np.float32(image)
    F_obraz = cv2.dft(image_float, flags=cv2.DFT_COMPLEX_OUTPUT)
    F_obraz_shift = np.fft.fftshift(F_obraz,[0,1])
    magnitude, phase = cv2.cartToPolar(F_obraz_shift[:,:,0], F_obraz_shift[:,:,1])

    return F_obraz_shift, magnitude, phase

def fourier_log(image):
    image_float = np.float32(image)
    F_obraz = cv2.dft(image_float, flags=cv2.DFT_COMPLEX_OUTPUT)
    F_obraz_shift = np.fft.fftshift(F_obraz,[0,1])
    magnitude, phase = cv2.cartToPolar(F_obraz_shift[:,:,0], F_obraz_shift[:,:,1])
    magnitude_log = np.log10(magnitude + 1)

    return F_obraz_shift, magnitude_log, phase

def fourrier_row_col(image):
    image_float = np.float32(image)
    F_row = np.fft.fft(image_float, axis=0)
    F_obraz = np.fft.fft(F_row, axis=1)
    F_shift = np.fft.fftshift(F_obraz, axes=[0,1])
    
    F_cv_format = np.stack((F_shift.real, F_shift.imag), axis=-1)
    magnitude, phase = cv2.cartToPolar(F_cv_format[:,:,0], F_cv_format[:,:,1])
    
    return F_cv_format, magnitude, phase


In [ ]:
image = cv2.imread("dwieFale.bmp", cv2.IMREAD_GRAYSCALE)

F_shift, mag_log, phase = fourier_log(image)

plot_3(image, mag_log, phase,
       "Oryginał", "Amplituda (log)", "Faza")

In [ ]:
_, mag, phase = fourier_not_log(image)

plot_3(image, mag, phase,
       "Oryginał", "Amplituda", "Faza")

In [ ]:
images = ["kolo.bmp", "kwadrat.bmp", "kwadrat45.bmp", "trojkat.bmp"]

for name in images:
    img = cv2.imread(name, cv2.IMREAD_GRAYSCALE)
    _, mag_log, phase = fourier_log(img)
    
    plot_3(img, mag_log, phase,
           f"{name}", "Amplituda (log)", "Faza")

In [ ]:
image = cv2.imread("dwieFale.bmp", cv2.IMREAD_GRAYSCALE)

F_cv, mag_cv, phase_cv = fourier_not_log(image)

F_np, mag_np, phase_np = fourrier_row_col(image)

diff_mag = np.abs(mag_cv - mag_np)
print("Max różnica (amplituda):", np.max(diff_mag))

diff = cv2.absdiff(F_cv, F_np)
print("Max różnica (Re/Im):", np.max(diff))

plot_3(mag_cv, mag_np, diff_mag,
       "Mag cv2.dft", "Mag fft", "Różnica")

Dwuwymiarowa transformata Fouriera umożliwia analizę obrazu w dziedzinie częstotliwości poprzez rozkład na amplitudę i fazę. Zastosowanie fftshift ułatwia interpretację widma, a logarytmowanie amplitudy poprawia jego czytelność.

Analiza F-obrazu pokazuje, że kierunki krawędzi w obrazie odpowiadają prostopadłym kierunkom w widmie częstotliwościowym.

Porównanie wyników cv2.dft oraz FFT wykonywanej kolejno po wierszach i kolumnach potwierdza, że dwuwymiarowa transformata Fouriera jest równoważna złożeniu dwóch jednowymiarowych transformat.

## Własności dwuwymiarowej transformaty Fouriera

1. Zbadaj jak zmienia się F-obraz (amplituda i faza) podczas następujących operacji: translacja, rotacja, zmiana rozmiaru, kombinacja liniowa.
Wykorzystaj stworzony wcześniej kod.<br>
Uwaga. Należy użyć przygotowanych obrazów, a nie "generować" własne.
2. Do badania translacji wykorzystaj obrazy *kwadrat.bmp* i *kwadratT.bmp*.
3. Przy badaniu rotacji wykorzystaj obrazy *kwadrat.bmp* i *kwadrat45.bmp*.
4. Przy badaniu zmiany rozmiaru wykorzystaj obrazy *kwadrat.bmp* i *kwadratS.bmp*.
5. Przy badaniu kombinacji liniowej wykorzystaj obrazy *kwadrat.bmp*, *kwadrat45.bmp* i *kwadratKL.bmp*.

In [ ]:
I_kwadrat = cv2.imread("kwadrat.bmp", cv2.IMREAD_GRAYSCALE)
I_kwadratT = cv2.imread("kwadratT.bmp", cv2.IMREAD_GRAYSCALE)

f_fcn_kwa,mag_kwa,pha_kwa=fourier_log(I_kwadrat)
f_fcn_kwaT,mag_kwaT,pha_kwaT=fourier_log(I_kwadratT)

plot_3(I_kwadrat,mag_kwa,pha_kwa)
plot_3(I_kwadratT,mag_kwaT,pha_kwaT)

In [ ]:
I_kwadrat45 = cv2.imread("kwadrat45.bmp", cv2.IMREAD_GRAYSCALE)

f_fcn_kwa,mag_kwa,pha_kwa=fourier_log(I_kwadrat)
f_fcn_kwa45,mag_kwa45,pha_kwa45=fourier_log(I_kwadrat45)

plot_3(I_kwadrat,mag_kwa,pha_kwa)
plot_3(I_kwadrat45,mag_kwa45,pha_kwa45)

In [ ]:
I_kwadratS = cv2.imread("kwadratS.bmp", cv2.IMREAD_GRAYSCALE)

f_fcn_kwa,mag_kwa,pha_kwa=fourier_log(I_kwadrat)
f_fcn_kwaS,mag_kwaS,pha_kwaS=fourier_log(I_kwadratS)

plot_3(I_kwadrat,mag_kwa,pha_kwa)
plot_3(I_kwadratS,mag_kwaS,pha_kwaS)

In [ ]:
I_kwadratKL = cv2.imread("kwadratKL.bmp", cv2.IMREAD_GRAYSCALE)

f_fcn_kwa,mag_kwa,pha_kwa=fourier_log(I_kwadrat)
f_fcn_kwa45,mag_kwa45,pha_kwa45=fourier_log(I_kwadrat45)
f_fcn_kwaKL,mag_kwaKL,pha_kwaKL=fourier_log(I_kwadratKL)

plot_3(I_kwadrat,mag_kwa,pha_kwa)
plot_3(I_kwadrat45,mag_kwa45,pha_kwa45)
plot_3(I_kwadratKL,mag_kwaKL,pha_kwaKL)

## Odwrotna dwuwymiarowa transformata Fouriera

1. Wykorzystaj stworzony wcześniej kod. Wybierz dowolny obraz np "kolo.bmp".
2. Przed realizacją odwrotnego przekszałcenia należy wykonać odwrotne przesunięcie.
Wykorzystaj funkcję `np.fft.ifftshift`.
Pierwszym argumentem jest wynik transformaty Fouriera.
Drugim argumentem są numery osi, wzdłuż których należy wykonać operację.
3. Wykonaj odwrotną transformatę Fouriera za pomocą funkcji `cv2.idft`.
Jako drugi argument przekaż następujące flagi: `flags=cv2.DFT_SCALE | cv2.DFT_COMPLEX_OUTPUT`.
Wynik może mieć małą część urojoną przez błędy numeryczne.
Aby pozbyć się tego efekty należy obliczyć amplitudę:
        `imgIFFT = cv2.magnitude(ifft[:, :, 0], ifft[:, :, 1])`
Następnie wynik należy zaokrąglić (`np.round`) i zrzutować do typu `uint8`.
4. Wyświetl wynik.
Sprawdź (wizualnie i poprzez odjęcie) czy obraz oryginalny i po przekształceniach są takie same.

In [ ]:
I_kolo = cv2.imread("kolo.bmp", cv2.IMREAD_GRAYSCALE)
f_fcn_kolo,mag_kolo,pha_kolo=fourier_log(I_kolo)
plot_3(I_kolo, mag_kolo, pha_kolo,
       "Koło", "Amplituda (log)", "Faza")

def inverse_fourier(F_shift):
    F_ishift = np.fft.ifftshift(F_shift, axes=[0,1])
    ifft = cv2.idft(F_ishift, flags=cv2.DFT_SCALE | cv2.DFT_COMPLEX_OUTPUT)
    img_back = cv2.magnitude(ifft[:,:,0], ifft[:,:,1])
    img_odwr= np.round(img_back).astype('uint8')
    return img_odwr

I_kolo_odwr = inverse_fourier(f_fcn_kolo)
plot_3(I_kolo, I_kolo_odwr, np.abs(I_kolo - I_kolo_odwr),
       "Oryginał", "Odwrotna DFT", "Różnica")


## Filtracja obrazu w dziedzinie częstotliwości

1. Wczytaj obraz "lena.bmp" w skali szarości.
Wykonaj transformatę Fouriera.
Wykorzystaj stworzony poprzednio kod.
Wyświetl obraz oryginalny, amplitudę (w skali logarytmicznej) i fazę.

2. Przeprowadź filtrację dolnoprzepustową - usuń górne częstotliwości.
Dla F-obrazu po operacji przesunięcia (`fftshift`) niskie częstotliwości leżą w jego centrum.

3. Na początku stwórz filtr "kołowy", dolnoprzepustowy.
Należy wygenerować macierze opisujące przestrzeń w dziedzinie częstotliwości.
Ich rozmiar musi być taki sam jak rozmiar przetwarzanego obrazu.

        lenaSize = I_Lena.shape
        FSpaceRows = 2 * np.fft.fftshift(np.fft.fftfreq(lenaSize[0]))
        FSpaceRowsM = np.outer(FSpaceRows, np.ones([1, lenaSize[1]]))
        FSpaceCols = 2 * np.fft.fftshift(np.fft.fftfreq(lenaSize[1]))
        FSpaceColsM = np.outer(np.ones([1, lenaSize[0]]), FSpaceCols)
        
Powyższy kod wygeneruje dwie znormalizowane macierze częstotliwości: *FSpaceRowsM* i *FSpaceColsM*.
Następnie należy wyznaczyć macierz zawierającą "odległość" od składowej stałej.
        `FreqR = np.sqrt(np.square(FSpaceRowsM) + np.square(FSpaceColsM))`

Uwagi:
- funkcja `fftfreq` generuje wektor częstotliwości $[-0.5, 0.5]$ o określonym rozmiarze, przy czym układ wartości jest taki, że najpierw od 0 do 0.5, a potem od -0.5 do 0,
- operacja `fftshift` zmienia ten układ na $[-0.5, 0.5]$,
- mnożenie przez 2 ustala ostatecznie zakres na $[-1, 1]$,
- operacja `outer` to tzw. iloczyn zewnętrzy dwóch wektorów, w naszym przypadku powoduje, że wektor pionowy lub poziomy jest "powielany" odpowiednią liczbę razy.   
- sugeruje się, aby przyglądnąć się jak wygląda macierz `FreqR` - czy to w debugerze, czy poprzez wizualizację.

4. Teraz należy wybrać interesujący zakres.
Tu można zdefiniować typ filtru (dolno, górno, pasmowoprzepustowy).

        FilterF = FreqR <= 0.1 

Filtr należy zwizualizować:

        figFilter = plt.figure()
        axsFilter = figFilter.add_subplot(projection='3d')
        axsFilter.plot_surface(FSpaceRowsM, FSpaceColsM, FilterF, rstride=3, cstride=3, cmap=plt.get_cmap('gray'), linewidth=0)
        figFilter.show()

4. Wykonaj właściwą filtrację, czyli mnożenie F-obrazu przez filtr FilterF.
Trzeba pamiętać, że F-obraz ma 2 kanały (rzeczywisty i urojony).
By mnożenie było możliwe należy więc powielić filtr również na 2 kanały.

        FilterF3 = np.repeat(FilterF[:, :, np.newaxis], 2, axis=2)

5. Wykonaj operację odwrotnego przesunięcia i odwrotnej transformaty.
Oblicz wartość bezwzględną wyniku.
Wykorzystaj funkcję `cv2.magnitude`.
Pierwszym argumentem jest część rzeczywista.
Drugim argumentem jest część urojona.
Wynik wyświetl.

6. Poeksperymentuj z rozmiarem filtru (promieniem).
Zaimplementuj filtr górnoprzepustowy (zmiana znaku przy warunku na odległość) oraz pasmowoprzepustowy (dwa warunki na promień połączone operatorem AND '&' ).
Wykonaj co najmniej trzy filtry i wyświetl wyniki.

7. W ten sposób zaimplementowana filtracja wprowadza pewne artefakty w postaci "pierścieni" wokół krawędzi.
Zapobiec temu zjawisku można poprzez odpowiednie "modelowanie" filtra.
W tym celu wykorzystać należy okna, np. Hamminga, Hanninga, Chebysheva (znane z przetwarzania sygnałów 1D).
Zagadnienie to jest tematem zadania domowego do tego ćwiczenia.

In [ ]:
I_lena = cv2.imread("lena.bmp", cv2.IMREAD_GRAYSCALE)

F_shift, mag_log, phase = fourier_log(I_lena)

plot_3(I_lena, mag_log, phase,
       "Lena", "Amplituda (log)", "Faza")

lenaSize = I_lena.shape

FSpaceRows = 2 * np.fft.fftshift(np.fft.fftfreq(lenaSize[0]))
FSpaceRowsM = np.outer(FSpaceRows, np.ones(lenaSize[1]))

FSpaceCols = 2 * np.fft.fftshift(np.fft.fftfreq(lenaSize[1]))
FSpaceColsM = np.outer(np.ones(lenaSize[0]), FSpaceCols)

FreqR = np.sqrt(FSpaceRowsM**2 + FSpaceColsM**2)

r = 0.1   # promień 

FilterF = FreqR <= r



In [ ]:
figFilter = plt.figure()
axsFilter = figFilter.add_subplot(projection='3d')
axsFilter.plot_surface(FSpaceRowsM, FSpaceColsM, FilterF, rstride=3, cstride=3, cmap=plt.get_cmap('gray'), linewidth=0)
figFilter.show()

In [ ]:
FilterF3 = np.repeat(FilterF[:, :, np.newaxis], 2, axis=2)
F_filtered = F_shift * FilterF3
I_filtered = inverse_fourier(F_filtered)
plot_3(I_lena, I_filtered, np.abs(I_lena - I_filtered),
       "Oryginał", "Po filtracji LP", "Różnica")

## Implementacja wyszukiwania wzorca za pomocą FFT

1. Wczytaj w skali szarości i wyświetl obrazy *literki.bmp* i *wzorA.bmp*.

2. Wyznacz transformatę Fouriera obrazu *literki.bmp*.

3. Obróć drugi obraz o $180^\circ$.
Zastosuj funkcję `np.rot90`.
Pierwszym argumentem jest obracana macierz, a drugim liczba obrotów o $90^\circ$.

4. Należy wyznaczyć transformatę Fouriera obróconego obrazu w taki sposób, żeby miała ona taki sam rozmiar jak pierwszy obraz.
W tym celu należy zastosować *Zero Padding*.
Operacja ta polega na uzupełnieniu obrazu zerami do oczekiwanego rozmiaru.
Uzupełnij obraz zerami z **prawej** strony i z **dołu**.
W tym celu należy wykorzystać funkcję `cv2.copyMakeBorder`.
    - Pierwszym argumentem jest obraz wejściowy.
    - Drugim argumentem jest liczba wierszy u góry.
    - Trzecim argumentem jest liczba wierszy u dołu.
    - Czwartym argumentem jest liczba kolumn z lewej.
    - Piątym argumentem jest liczba kolumn z prawej.
    - Szóstym argumentem jest flaga typu wypełnienia.
    Dla stałej wartości podaj `cv2.BORDER_CONSTANT`.
    - Siódmym argumentem jest wartość pikseli w ramce.
    Przekaż `value=0`.

5. Wyznacz transformatę Fouriera obrazu stworzonego w poprzednim punkcie.

6. Wyniki obu transformat należy przekonwertować do liczb zespolonych.
Obecnie jest to dwukanałowa macierz.
Pierwszy kanał odpowiada za część rzeczywistą.
Drugi kanał odpowiada za część urojoną.
Aby to osiągnąć wystarczy wykonać działanie:

        Complex = Real + Imag * 1j

7. Przemnóż ze sobą zespolone wyniki transformat.

8. Wynik należy powrotnie przekształcić do dwukanałowej macierzy.
Aby to zrobić wykonaj operację:

        CompMat = cv2.merge([np.real(Complex), np.imag(Complex)])

9. Wykonaj odwrotną transformatę Fouriera.
Dodaj flagę `flags=cv2.DFT_COMPLEX_INPUT`.

10. Oblicz wartość bezwzględną wyniku.

11. Wykonaj morfologiczną operację **Top-Hat**, by znaleźć maksima lokalne.
Operacja ta zostanie dokładnej wyjaśniona w jednym z kolejnych ćwiczeń.
W tym celu wykorzystaj operację:

        cv2.morphologyEx(correlation, cv2.MORPH_TOPHAT, np.ones((3, 3), np.uint8))

12. Wyświetl obok siebie obraz wejściowy i wynik wykonanych operacji.
Czy możesz wskazać położenie wzoru na podstawie drugiego obrazu?

In [ ]:
I = cv2.imread("literki.bmp", cv2.IMREAD_GRAYSCALE)
T = cv2.imread("wzorA.bmp", cv2.IMREAD_GRAYSCALE)

plot_3(I, T, I, "Obraz", "Wzorzec", "")

I_float = np.float32(I)
F_I = cv2.dft(I_float, flags=cv2.DFT_COMPLEX_OUTPUT)
T_rot = np.rot90(T, 2)

hI, wI = I.shape
hT, wT = T_rot.shape

T_pad = cv2.copyMakeBorder(
    T_rot,
    0, hI - hT,     # góra, dół
    0, wI - wT,     # lewo, prawo
    cv2.BORDER_CONSTANT,
    value=0
)

T_float = np.float32(T_pad)
F_T = cv2.dft(T_float, flags=cv2.DFT_COMPLEX_OUTPUT)

F_I_complex = F_I[:,:,0] + 1j * F_I[:,:,1]
F_T_complex = F_T[:,:,0] + 1j * F_T[:,:,1]

F_corr = F_I_complex * F_T_complex

F_corr_cv = cv2.merge([np.real(F_corr), np.imag(F_corr)])

corr = cv2.idft(F_corr_cv, flags=cv2.DFT_COMPLEX_INPUT | cv2.DFT_SCALE)

corr_mag = cv2.magnitude(corr[:,:,0], corr[:,:,1])

kernel = np.ones((3,3), np.uint8)
corr_tophat = cv2.morphologyEx(corr_mag, cv2.MORPH_TOPHAT, kernel)

plot_3(I, corr_mag, corr_tophat,
       "Obraz", "Korelacja", "Top-Hat (maksima)")

Można wskazać położenie wzorca na podstawie wyniku, tam, gdzie znajdują się wyraźne białe kropki, tam jest litera A